# Bible Verse & Scripture Explainer

A multi-modal AI assistant that helps you understand Scripture like a theologian or preacher.

Enter a Bible verse (e.g. *John 3:16*, *Romans 8:28*) or paste the text, or hit **Random** to get a random verse.
The assistant will:
- **Explain** the verse with theological depth, context, and practical application
- **Draw** a powerful illustration inspired by the scripture (DALL-E)
- **Narrate** a sermon summary


In [ ]:
import os
import json
import random
import base64
from io import BytesIO
from dotenv import load_dotenv
from openai import OpenAI
from PIL import Image
import gradio as gr

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

MODEL = "gpt-4.1-mini"
openai = OpenAI()

## Verse Bank

A curated collection of well-known Bible verses. The LLM uses tool calling to fetch a random verse when the user clicks **Surprise Me**, then explains it in theologian/preacher style.

In [ ]:
VERSES = [
    {"reference": "John 3:16", "text": "For God so loved the world that he gave his one and only Son, that whoever believes in him shall not perish but have eternal life."},
    {"reference": "Romans 8:28", "text": "And we know that in all things God works for the good of those who love him, who have been called according to his purpose."},
    {"reference": "Philippians 4:13", "text": "I can do all this through him who gives me strength."},
    {"reference": "Jeremiah 29:11", "text": "For I know the plans I have for you, declares the Lord, plans to prosper you and not to harm you, plans to give you hope and a future."},
    {"reference": "Psalm 23:1", "text": "The Lord is my shepherd, I lack nothing."},
    {"reference": "Proverbs 3:5-6", "text": "Trust in the Lord with all your heart and lean not on your own understanding; in all your ways submit to him, and he will make your paths straight."},
    {"reference": "Isaiah 41:10", "text": "So do not fear, for I am with you; do not be dismayed, for I am your God. I will strengthen you and help you; I will uphold you with my righteous right hand."},
    {"reference": "Matthew 11:28", "text": "Come to me, all you who are weary and burdened, and I will give you rest."},
    {"reference": "Romans 12:2", "text": "Do not conform to the pattern of this world, but be transformed by the renewing of your mind. Then you will be able to test and approve what God's will is—his good, pleasing and perfect will."},
    {"reference": "2 Corinthians 5:7", "text": "For we live by faith, not by sight."},
]

print(f"Loaded {len(VERSES)} verses in the verse bank")

In [ ]:
def get_random_verse():
    """Return a random Bible verse from the verse bank."""
    verse = random.choice(VERSES)
    print(f"TOOL CALLED: random verse {verse['reference']}", flush=True)
    return json.dumps(verse)


random_verse_schema = {
    "name": "get_random_verse",
    "description": "Pick a random Bible verse from the curated collection. Call this when the user wants a surprise verse or says 'surprise me'.",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": [],
        "additionalProperties": False,
    },
}

tools = [
    {"type": "function", "function": random_verse_schema},
]

## System Prompt & Agent Logic

The assistant plays the role of a warm, knowledgeable theologian or preacher
who explains Scripture with depth, context, and practical application.

In [ ]:
system_message = """You are a warm, knowledgeable theologian and preacher who explains Bible verses with depth and pastoral care.

When given a Bible verse (by reference like "John 3:16" or by pasted text), follow this structure:

1. State the verse reference and the full text of the verse.
2. Briefly set the context: book, author, original audience, and place in the biblical story.
3. Explain the meaning theologically — what God is revealing, promising, or commanding.
4. Offer a practical application: how this truth can shape our faith, choices, or comfort today.
5. End with a short, encouraging reflection or closing thought a preacher might use.

Keep the tone conversational yet reverent — like a thoughtful pastor unpacking Scripture for the congregation.
If the user says "surprise me" or similar, call get_random_verse and then explain that verse using the structure above.
For any other verse the user enters (reference or text), explain it directly using the same structure."""

In [ ]:
def handle_tool_calls(message):
    """Execute every tool call the model requested and return results."""
    responses = []
    for tool_call in message.tool_calls:
        name = tool_call.function.name
        args = json.loads(tool_call.function.arguments) if tool_call.function.arguments else {}
        if name == "get_random_verse":
            result = get_random_verse()
        else:
            result = json.dumps({"error": f"Unknown tool: {name}"})
        responses.append({
            "role": "tool",
            "content": result,
            "tool_call_id": tool_call.id,
        })
    return responses


def artist(verse_text):
    """Generate a DALL-E illustration inspired by the Bible verse."""
    image_response = openai.images.generate(
        model="dall-e-3",
        prompt=(
            f"A reverent, dignified illustration inspired by this Bible verse: "
            f"'{verse_text}'. "
            f"Style: warm light, classical or contemplative mood, "
            f"appropriate for Scripture — no text in the image."
        ),
        size="1024x1024",
        n=1,
        response_format="b64_json",
    )
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))


def talker(text):
    """Generate TTS narration of the given text."""
    response = openai.audio.speech.create(
        model="gpt-4o-mini-tts",
        voice="onyx",
        input=text,
    )
    return response.content

In [ ]:
def chat(history):
    """Agent loop: tool calls -> explanation -> illustration -> narration."""
    messages = [{"role": "system", "content": system_message}]
    messages += [{"role": h["role"], "content": h["content"]} for h in history]

    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason == "tool_calls":
        assistant_msg = response.choices[0].message
        tool_results = handle_tool_calls(assistant_msg)
        messages.append(assistant_msg)
        messages.extend(tool_results)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    reply = response.choices[0].message.content
    history.append({"role": "assistant", "content": reply})

    voice = talker(reply)

    first_line = reply.split("\n")[0]
    image = artist(first_line)

    return history, voice, image

## Gradio Interface

A `gr.Blocks` layout with a chatbot panel, illustration panel, and audio narration.

In [ ]:
def add_user_message(message, history):
    """Move user message into chatbot history and clear the textbox."""
    return "", history + [{"role": "user", "content": message}]


def surprise_me(history):
    """Inject a 'surprise me' request into the chat."""
    return history + [{"role": "user", "content": "Surprise me with a random Bible verse!"}]


with gr.Blocks(title="Bible Verse Explainer") as ui:
    gr.Markdown("# Bible Verse & Scripture Explainer")
    gr.Markdown(
        "Enter a Bible verse (e.g. *John 3:16*, *Romans 8:28*) or paste the text. "
        "Click **Random** to get a random verse explained like a theologian or preacher."
    )
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        image_output = gr.Image(height=500, label="Illustration", interactive=False)
    with gr.Row():
        audio_output = gr.Audio(label="Narration", autoplay=True)
    with gr.Row():
        message = gr.Textbox(
            label="Enter a Bible verse:",
            placeholder="e.g. John 3:16, Romans 8:28, or paste the verse text...",
        )
        surprise_btn = gr.Button("Random")

    message.submit(add_user_message, [message, chatbot], [message, chatbot]).then(
        chat, chatbot, [chatbot, audio_output, image_output]
    )
    surprise_btn.click(surprise_me, chatbot, chatbot).then(
        chat, chatbot, [chatbot, audio_output, image_output]
    )

ui.launch(inbrowser=True)